# Convert PyTorch ckpts → OpenVINO IR (7 models batch)

## Models
1. **exp015 R2** (5s convnext_pico, Tucker mel)
2. **exp017 R2** (5s eca_nfnet_l0, Tucker mel)
3. **exp016 R2** (5s regnety_008, Tucker mel) ★ NEW
4. **exp081 R2** (10s eca_nfnet_l0, Tucker mel)
5. **exp082 R2** (20s eca_nfnet_l0, Babych mel)
6. **exp029 R3** (5s eca_nfnet_l1, Tucker mel, fold 0) ★ NEW

## Output (per model, into existing Dataset's subfolder)
- `r2_ov/model.xml` + `r2_ov/model.bin` for R2 models
- **`r3_ov/model.xml` + `r3_ov/model.bin`** for exp029 R3 (note r3, not r2)

## Constraints
- Internet=True (openvino install + Kaggle upload)
- CPU or GPU mode (CPU OK for conversion)
- Conversion time: ~3-5 min/model, total ~25-40 min

## Inference NB usage (post-conversion)
```python
import openvino as ov
core = ov.Core()
model = core.read_model('/kaggle/input/datasets/maekeso/birdclef2026-exp029-l1-single/r3_ov/model.xml')
compiled = core.compile_model(model, 'CPU')
result = compiled.create_infer_request().infer({input_port: mel_np})
```


In [ ]:
# ============================================================
# Cell 1: Setup — install openvino + imports
# ============================================================
!pip install -q openvino==2024.4.0 onnx

import os, json, shutil, tempfile, time
from pathlib import Path
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import openvino as ov

print(f'torch={torch.__version__}, timm={timm.__version__}, ov={ov.__version__}')

# Kaggle API auth (Kaggle env auto-loads /kaggle/secrets if set)
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
print('Kaggle authenticated OK')


In [ ]:
# ============================================================
# Cell 2: Model class (BirdSEDModel) + Wrapper for ONNX export
# ============================================================
NUM_CLASSES = 234
PERCH_EMBED_DIM = 1536
USE_PERCH_DISTILL = True

class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        return self.proj(feature_map.mean(dim=[2, 3]))


class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name, num_classes=NUM_CLASSES,
                 drop_path_rate=0.0, hidden_dim=512, n_mels=256, time_frames=313):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=False, in_chans=1,
            num_classes=0, global_pool='', drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            dummy = torch.randn(1, 1, n_mels, time_frames)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=True):
        h = self.backbone(x)
        h_cls = self.gem_freq(h)
        h_cls = h_cls.permute(0, 2, 1)
        h_cls = self.dense(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise, dim=2)
        # Return both for OV (single output is harder to specify)
        return clip_logits, framewise.permute(0, 2, 1)


class OVWrapper(nn.Module):
    '''Thin wrapper: always returns (clip_logits, framewise) for clean OV export.'''
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, mel):
        return self.model(mel)

print('OK model class ready')


In [ ]:
# ============================================================
# Cell 3: Helpers — convert + upload (robust multi-path)
# ============================================================
ONNX_OUT_ROOT = Path('/kaggle/working/ov_exports')
ONNX_OUT_ROOT.mkdir(parents=True, exist_ok=True)


def convert_pytorch_to_ov(ckpt_path, backbone_name, n_mels, time_frames,
                          model_name, drop_path_rate=0.0, hidden_dim=512,
                          compress_fp16=False):
    """Convert PyTorch ckpt → OpenVINO IR with robust multi-path fallback.

    Try order:
    1. ov.convert_model(pytorch_model, example_input=dummy) — direct
    2. PyTorch → ONNX opset 14 → OV
    3. PyTorch → ONNX opset 17 → OV
    """
    print(f'\n--- {model_name} ---')
    print(f'  ckpt: {ckpt_path}')
    print(f'  backbone: {backbone_name}, mel ({n_mels}, {time_frames})')

    # Load PyTorch state
    state = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    epoch = state.get('epoch', '?')
    ns22 = state.get('best_ns22', float('nan'))
    macro = state.get('best_macro', float('nan'))
    print(f'  ckpt epoch={epoch}, best_ns22={ns22:.4f}, best_macro={macro:.4f}')

    # Build model
    model = BirdSEDModel(backbone_name=backbone_name, n_mels=n_mels,
                         time_frames=time_frames, drop_path_rate=drop_path_rate,
                         hidden_dim=hidden_dim).eval()
    msg = model.load_state_dict(state['model_state'], strict=False)
    print(f'  load: missing={len(msg.missing_keys)}, unexpected={len(msg.unexpected_keys)}')
    if len(msg.missing_keys) > 0:
        print(f'    missing sample: {list(msg.missing_keys)[:3]}')
    if len(msg.unexpected_keys) > 0:
        print(f'    unexpected sample: {list(msg.unexpected_keys)[:3]}')
    assert len(msg.missing_keys) < 5, f'Too many missing keys: {len(msg.missing_keys)}'

    # Wrap for export
    wrapper = OVWrapper(model).eval()

    # Test forward to verify
    dummy = torch.randn(1, 1, n_mels, time_frames)
    with torch.no_grad():
        try:
            test_out = wrapper(dummy)
            print(f'  Test forward OK: clip {test_out[0].shape}, framewise {test_out[1].shape}')
        except Exception as e:
            raise RuntimeError(f'PyTorch forward failed: {type(e).__name__}: {e}')

    # Output dir
    out_dir = ONNX_OUT_ROOT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # ============================================================
    # Path 1: ov.convert_model direct from PyTorch (most robust)
    # ============================================================
    print(f'\n  [Path 1] Trying PyTorch direct → OV...')
    ov_model = None
    try:
        t0 = time.time()
        ov_model = ov.convert_model(wrapper, example_input=dummy)
        print(f'    [OK] Direct convert succeeded ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'    [FAIL] {type(e).__name__}: {str(e)[:300]}')

    # ============================================================
    # Path 2: PyTorch → ONNX opset 14 → OV
    # ============================================================
    if ov_model is None:
        print(f'\n  [Path 2] Trying PyTorch → ONNX opset 14 → OV...')
        onnx_path = out_dir / 'model.onnx'
        try:
            t0 = time.time()
            torch.onnx.export(
                wrapper, dummy, str(onnx_path),
                input_names=['mel'],
                output_names=['clip_logits', 'framewise'],
                dynamic_axes={
                    'mel': {0: 'batch'},
                    'clip_logits': {0: 'batch'},
                    'framewise': {0: 'batch'},
                },
                opset_version=14,
                do_constant_folding=True,
                dynamo=False,
            )
            print(f'    ONNX (opset 14) exported ({time.time()-t0:.1f}s, {onnx_path.stat().st_size/1e6:.1f}MB)')
            t0 = time.time()
            ov_model = ov.convert_model(str(onnx_path))
            print(f'    [OK] OV converted ({time.time()-t0:.1f}s)')
            onnx_path.unlink()
        except Exception as e:
            print(f'    [FAIL] {type(e).__name__}: {str(e)[:300]}')
            if onnx_path.exists():
                onnx_path.unlink()

    # ============================================================
    # Path 3: PyTorch → ONNX opset 17 → OV
    # ============================================================
    if ov_model is None:
        print(f'\n  [Path 3] Trying PyTorch → ONNX opset 17 → OV...')
        onnx_path = out_dir / 'model.onnx'
        try:
            t0 = time.time()
            torch.onnx.export(
                wrapper, dummy, str(onnx_path),
                input_names=['mel'],
                output_names=['clip_logits', 'framewise'],
                dynamic_axes={
                    'mel': {0: 'batch'},
                    'clip_logits': {0: 'batch'},
                    'framewise': {0: 'batch'},
                },
                opset_version=17,
                do_constant_folding=True,
                dynamo=False,
            )
            print(f'    ONNX (opset 17) exported ({time.time()-t0:.1f}s)')
            t0 = time.time()
            ov_model = ov.convert_model(str(onnx_path))
            print(f'    [OK] OV converted ({time.time()-t0:.1f}s)')
            onnx_path.unlink()
        except Exception as e:
            print(f'    [FAIL] {type(e).__name__}: {str(e)[:300]}')
            if onnx_path.exists():
                onnx_path.unlink()

    if ov_model is None:
        raise RuntimeError(f'All 3 conversion paths failed for {model_name}')

    # Save OV IR
    ov_xml = out_dir / 'model.xml'
    ov.save_model(ov_model, str(ov_xml), compress_to_fp16=compress_fp16)
    ov_bin = out_dir / 'model.bin'
    print(f'\n  OV IR saved:')
    print(f'    {ov_xml.name}: {ov_xml.stat().st_size/1024:.1f}KB')
    print(f'    {ov_bin.name}: {ov_bin.stat().st_size/1e6:.1f}MB')

    return out_dir


def upload_ov_to_dataset(dataset_slug, ov_local_dir, ov_subfolder, version_notes):
    """Upload OV IR to existing Kaggle Dataset, preserving existing files."""
    print(f'\n  Uploading to {dataset_slug} → {ov_subfolder}/')
    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        # ★ Kaggle Dataset mount: try datasets/maekeso/<slug> first
        existing = None
        for cand_existing in [
            Path(f'/kaggle/input/datasets/maekeso/{dataset_slug}'),
            Path(f'/kaggle/input/{dataset_slug}'),
        ]:
            if cand_existing.exists():
                existing = cand_existing
                break

        if existing is None:
            print(f'    [WARN] dataset not attached, will create fresh')
            n_preserved = 0
        else:
            n_preserved = 0
            for f in existing.rglob('*'):
                if not f.is_file(): continue
                if f.name.startswith('__'): continue
                rel = f.relative_to(existing)
                dst = td / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(str(f), str(dst))
                n_preserved += 1
            print(f'    Preserved {n_preserved} existing files')

        # Add OV files in subfolder
        ov_dst = td / ov_subfolder
        ov_dst.mkdir(parents=True, exist_ok=True)
        for f in ov_local_dir.iterdir():
            if f.is_file():
                shutil.copy2(str(f), str(ov_dst / f.name))
                print(f'    Added: {ov_subfolder}/{f.name} ({f.stat().st_size/1e6:.1f}MB)')

        # Metadata
        meta = {
            'title': dataset_slug.replace('-', ' '),
            'id': f'maekeso/{dataset_slug}',
            'licenses': [{'name': 'CC0-1.0'}],
        }
        (td / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')

        # Upload
        try:
            t0 = time.time()
            api.dataset_create_version(
                folder=str(td), version_notes=version_notes,
                dir_mode='zip', quiet=False,
            )
            print(f'    OK Uploaded ({time.time()-t0:.0f}s)')
            return True
        except Exception as e:
            print(f'    FAILED: {type(e).__name__}: {str(e)[:300]}')
            return False


print('OK helpers ready (3-path robust converter + correct dataset mount)')


In [ ]:
# ============================================================
# Cell: Convert exp015 R2 (convnext_pico, 5s Tucker)
# ============================================================
print('='*60)
print('Converting exp015 R2 (convnext_pico, 5s Tucker)')
print('='*60)

dataset_slug = 'birdclef2026-exp015-weights'
ckpt_candidates = ['r2_ckpt_best_ns22.pth']

# Find ckpt (with rglob fallback for unknown structure)
# ★ Kaggle Dataset mount: /kaggle/input/datasets/<owner>/<slug>/
DATASET_ROOTS = [
    Path('/kaggle/input/datasets/maekeso') / dataset_slug,  # ★ correct mount
    Path('/kaggle/input') / dataset_slug,                    # fallback (古い)
]
dataset_root = None
for p in DATASET_ROOTS:
    if p.exists():
        dataset_root = p
        print(f'  dataset_root: {p}')
        break
assert dataset_root is not None, f'Dataset not mounted: tried {DATASET_ROOTS}'

ckpt_path = None
# Try direct paths first
for cand in ckpt_candidates:
    p = dataset_root / cand
    if p.exists():
        ckpt_path = p; break

# Fallback: rglob search (whole dataset tree)
if ckpt_path is None:
    target_name = Path(ckpt_candidates[0]).name
    print(f'  Direct paths not found, rglob fallback for {target_name}...')
    for f in dataset_root.rglob(target_name):
        ckpt_path = f
        print(f'    Found via rglob: {f}')
        break

# Debug: list dataset contents if still not found
if ckpt_path is None:
    print(f'  [DEBUG] {dataset_root} contents:')
    for f in sorted(dataset_root.rglob('*'))[:30]:
        if f.is_file():
            print(f'    {f.relative_to(dataset_root)} ({f.stat().st_size/1e6:.1f}MB)')

assert ckpt_path is not None, f'ckpt not found in {dataset_slug}'

# Convert
out_dir = convert_pytorch_to_ov(
    ckpt_path=ckpt_path,
    backbone_name='convnext_pico.d1_in1k',
    n_mels=256,
    time_frames=313,
    model_name='exp015_r2',
    drop_path_rate=0.0,
    hidden_dim=512,
    compress_fp16=False,
)

# Upload to Dataset (preserve existing + add r2_ov/)
ok = upload_ov_to_dataset(
    dataset_slug=dataset_slug,
    ov_local_dir=out_dir,
    ov_subfolder='r2_ov',
    version_notes='Added OpenVINO IR (r2_ov/)',
)
print(f'\n  Result: {"OK" if ok else "FAIL"}')


In [ ]:
# ============================================================
# Cell: Convert exp017 R2 (eca_nfnet_l0, 5s Tucker)
# ============================================================
print('='*60)
print('Converting exp017 R2 (eca_nfnet_l0, 5s Tucker)')
print('='*60)

dataset_slug = 'birdclef2026-exp017-weights'
ckpt_candidates = ['r2_ckpt_best_ns22.pth']

# Find ckpt (with rglob fallback for unknown structure)
# ★ Kaggle Dataset mount: /kaggle/input/datasets/<owner>/<slug>/
DATASET_ROOTS = [
    Path('/kaggle/input/datasets/maekeso') / dataset_slug,  # ★ correct mount
    Path('/kaggle/input') / dataset_slug,                    # fallback (古い)
]
dataset_root = None
for p in DATASET_ROOTS:
    if p.exists():
        dataset_root = p
        print(f'  dataset_root: {p}')
        break
assert dataset_root is not None, f'Dataset not mounted: tried {DATASET_ROOTS}'

ckpt_path = None
# Try direct paths first
for cand in ckpt_candidates:
    p = dataset_root / cand
    if p.exists():
        ckpt_path = p; break

# Fallback: rglob search (whole dataset tree)
if ckpt_path is None:
    target_name = Path(ckpt_candidates[0]).name
    print(f'  Direct paths not found, rglob fallback for {target_name}...')
    for f in dataset_root.rglob(target_name):
        ckpt_path = f
        print(f'    Found via rglob: {f}')
        break

# Debug: list dataset contents if still not found
if ckpt_path is None:
    print(f'  [DEBUG] {dataset_root} contents:')
    for f in sorted(dataset_root.rglob('*'))[:30]:
        if f.is_file():
            print(f'    {f.relative_to(dataset_root)} ({f.stat().st_size/1e6:.1f}MB)')

assert ckpt_path is not None, f'ckpt not found in {dataset_slug}'

# Convert
out_dir = convert_pytorch_to_ov(
    ckpt_path=ckpt_path,
    backbone_name='eca_nfnet_l0',
    n_mels=256,
    time_frames=313,
    model_name='exp017_r2',
    drop_path_rate=0.0,
    hidden_dim=512,
    compress_fp16=False,
)

# Upload to Dataset (preserve existing + add r2_ov/)
ok = upload_ov_to_dataset(
    dataset_slug=dataset_slug,
    ov_local_dir=out_dir,
    ov_subfolder='r2_ov',
    version_notes='Added OpenVINO IR (r2_ov/)',
)
print(f'\n  Result: {"OK" if ok else "FAIL"}')


In [ ]:
# ============================================================
# Cell: Convert exp016 R2 (regnety_008, 5s Tucker)
# ============================================================
print('='*60)
print('Converting exp016 R2 (regnety_008, 5s Tucker)')
print('='*60)

dataset_slug = 'birdclef2026-exp016-weights'
ckpt_candidates = ['r2_ckpt_best_ns22.pth']

DATASET_ROOTS = [
    Path('/kaggle/input/datasets/maekeso') / dataset_slug,
    Path('/kaggle/input') / dataset_slug,
]
dataset_root = None
for p in DATASET_ROOTS:
    if p.exists():
        dataset_root = p
        print(f'  dataset_root: {p}')
        break
assert dataset_root is not None, f'Dataset not mounted: tried {DATASET_ROOTS}'

ckpt_path = None
for cand in ckpt_candidates:
    p = dataset_root / cand
    if p.exists():
        ckpt_path = p; break

if ckpt_path is None:
    target_name = Path(ckpt_candidates[0]).name
    print(f'  Direct paths not found, rglob fallback for {target_name}...')
    for f in dataset_root.rglob(target_name):
        ckpt_path = f
        print(f'    Found via rglob: {f}')
        break

if ckpt_path is None:
    print(f'  [DEBUG] {dataset_root} contents:')
    for f in sorted(dataset_root.rglob('*'))[:30]:
        if f.is_file():
            print(f'    {f.relative_to(dataset_root)} ({f.stat().st_size/1e6:.1f}MB)')

assert ckpt_path is not None, f'ckpt not found in {dataset_slug}'

out_dir = convert_pytorch_to_ov(
    ckpt_path=ckpt_path,
    backbone_name='regnety_008',
    n_mels=256,
    time_frames=313,
    model_name='exp016_r2',
    drop_path_rate=0.0,
    hidden_dim=512,
    compress_fp16=False,
)

ok = upload_ov_to_dataset(
    dataset_slug=dataset_slug,
    ov_local_dir=out_dir,
    ov_subfolder='r2_ov',
    version_notes='Added OpenVINO IR (r2_ov/)',
)
print(f'\n  Result: {"OK" if ok else "FAIL"}')


In [ ]:
# ============================================================
# Cell: Convert exp081 R2 (eca_nfnet_l0, 10s Tucker)
# ============================================================
print('='*60)
print('Converting exp081 R2 (eca_nfnet_l0, 10s Tucker)')
print('='*60)

dataset_slug = 'birdclef2026-exp081-weights'
ckpt_candidates = ['r2/ckpt_best_ns22.pth']

# Find ckpt (with rglob fallback for unknown structure)
# ★ Kaggle Dataset mount: /kaggle/input/datasets/<owner>/<slug>/
DATASET_ROOTS = [
    Path('/kaggle/input/datasets/maekeso') / dataset_slug,  # ★ correct mount
    Path('/kaggle/input') / dataset_slug,                    # fallback (古い)
]
dataset_root = None
for p in DATASET_ROOTS:
    if p.exists():
        dataset_root = p
        print(f'  dataset_root: {p}')
        break
assert dataset_root is not None, f'Dataset not mounted: tried {DATASET_ROOTS}'

ckpt_path = None
# Try direct paths first
for cand in ckpt_candidates:
    p = dataset_root / cand
    if p.exists():
        ckpt_path = p; break

# Fallback: rglob search (whole dataset tree)
if ckpt_path is None:
    target_name = Path(ckpt_candidates[0]).name
    print(f'  Direct paths not found, rglob fallback for {target_name}...')
    for f in dataset_root.rglob(target_name):
        ckpt_path = f
        print(f'    Found via rglob: {f}')
        break

# Debug: list dataset contents if still not found
if ckpt_path is None:
    print(f'  [DEBUG] {dataset_root} contents:')
    for f in sorted(dataset_root.rglob('*'))[:30]:
        if f.is_file():
            print(f'    {f.relative_to(dataset_root)} ({f.stat().st_size/1e6:.1f}MB)')

assert ckpt_path is not None, f'ckpt not found in {dataset_slug}'

# Convert
out_dir = convert_pytorch_to_ov(
    ckpt_path=ckpt_path,
    backbone_name='eca_nfnet_l0',
    n_mels=256,
    time_frames=626,
    model_name='exp081_r2',
    drop_path_rate=0.0,
    hidden_dim=512,
    compress_fp16=False,
)

# Upload to Dataset (preserve existing + add r2_ov/)
ok = upload_ov_to_dataset(
    dataset_slug=dataset_slug,
    ov_local_dir=out_dir,
    ov_subfolder='r2_ov',
    version_notes='Added OpenVINO IR (r2_ov/)',
)
print(f'\n  Result: {"OK" if ok else "FAIL"}')


In [ ]:
# ============================================================
# Cell: Convert exp082 R2 (eca_nfnet_l0, 20s Babych)
# ============================================================
print('='*60)
print('Converting exp082 R2 (eca_nfnet_l0, 20s Babych)')
print('='*60)

dataset_slug = 'birdclef2026-exp082-weights'
ckpt_candidates = ['r2/ckpt_best_ns22.pth']

# Find ckpt (with rglob fallback for unknown structure)
# ★ Kaggle Dataset mount: /kaggle/input/datasets/<owner>/<slug>/
DATASET_ROOTS = [
    Path('/kaggle/input/datasets/maekeso') / dataset_slug,  # ★ correct mount
    Path('/kaggle/input') / dataset_slug,                    # fallback (古い)
]
dataset_root = None
for p in DATASET_ROOTS:
    if p.exists():
        dataset_root = p
        print(f'  dataset_root: {p}')
        break
assert dataset_root is not None, f'Dataset not mounted: tried {DATASET_ROOTS}'

ckpt_path = None
# Try direct paths first
for cand in ckpt_candidates:
    p = dataset_root / cand
    if p.exists():
        ckpt_path = p; break

# Fallback: rglob search (whole dataset tree)
if ckpt_path is None:
    target_name = Path(ckpt_candidates[0]).name
    print(f'  Direct paths not found, rglob fallback for {target_name}...')
    for f in dataset_root.rglob(target_name):
        ckpt_path = f
        print(f'    Found via rglob: {f}')
        break

# Debug: list dataset contents if still not found
if ckpt_path is None:
    print(f'  [DEBUG] {dataset_root} contents:')
    for f in sorted(dataset_root.rglob('*'))[:30]:
        if f.is_file():
            print(f'    {f.relative_to(dataset_root)} ({f.stat().st_size/1e6:.1f}MB)')

assert ckpt_path is not None, f'ckpt not found in {dataset_slug}'

# Convert
out_dir = convert_pytorch_to_ov(
    ckpt_path=ckpt_path,
    backbone_name='eca_nfnet_l0',
    n_mels=224,
    time_frames=512,
    model_name='exp082_r2',
    drop_path_rate=0.0,
    hidden_dim=512,
    compress_fp16=False,
)

# Upload to Dataset (preserve existing + add r2_ov/)
ok = upload_ov_to_dataset(
    dataset_slug=dataset_slug,
    ov_local_dir=out_dir,
    ov_subfolder='r2_ov',
    version_notes='Added OpenVINO IR (r2_ov/)',
)
print(f'\n  Result: {"OK" if ok else "FAIL"}')


In [ ]:
# ============================================================
# Cell: Convert exp029 R3 (eca_nfnet_l1, 5s Tucker, fold 0)
# ============================================================
print('='*60)
print('Converting exp029 R3 (eca_nfnet_l1, 5s Tucker, fold 0)')
print('='*60)

dataset_slug = 'birdclef2026-exp029-l1-single'
ckpt_candidates = ['r3_fold0_ckpt_best_ns22.pth']

DATASET_ROOTS = [
    Path('/kaggle/input/datasets/maekeso') / dataset_slug,
    Path('/kaggle/input') / dataset_slug,
]
dataset_root = None
for p in DATASET_ROOTS:
    if p.exists():
        dataset_root = p
        print(f'  dataset_root: {p}')
        break
assert dataset_root is not None, f'Dataset not mounted: tried {DATASET_ROOTS}'

ckpt_path = None
for cand in ckpt_candidates:
    p = dataset_root / cand
    if p.exists():
        ckpt_path = p; break

if ckpt_path is None:
    target_name = Path(ckpt_candidates[0]).name
    print(f'  Direct paths not found, rglob fallback for {target_name}...')
    for f in dataset_root.rglob(target_name):
        ckpt_path = f
        print(f'    Found via rglob: {f}')
        break

if ckpt_path is None:
    print(f'  [DEBUG] {dataset_root} contents:')
    for f in sorted(dataset_root.rglob('*'))[:30]:
        if f.is_file():
            print(f'    {f.relative_to(dataset_root)} ({f.stat().st_size/1e6:.1f}MB)')

assert ckpt_path is not None, f'ckpt not found in {dataset_slug}'

# ★ exp029 R3: backbone eca_nfnet_l1 (l0 vs l1 capacity 異)
out_dir = convert_pytorch_to_ov(
    ckpt_path=ckpt_path,
    backbone_name='eca_nfnet_l1',
    n_mels=256,
    time_frames=313,
    model_name='exp029_r3',
    drop_path_rate=0.0,
    hidden_dim=512,
    compress_fp16=False,
)

# ★ exp029 dataset は r3_ov/ subfolder にする (r2 でなく r3)
ok = upload_ov_to_dataset(
    dataset_slug=dataset_slug,
    ov_local_dir=out_dir,
    ov_subfolder='r3_ov',
    version_notes='Added OpenVINO IR (r3_ov/)',
)
print(f'\n  Result: {"OK" if ok else "FAIL"}')


In [ ]:
# ============================================================
# Cell: Summary
# ============================================================
print('\n' + '='*60)
print('Conversion summary')
print('='*60)
print('\nConverted OV files:')
for sub in sorted(ONNX_OUT_ROOT.iterdir()):
    if sub.is_dir():
        xml = sub / 'model.xml'
        bin_f = sub / 'model.bin'
        if xml.exists() and bin_f.exists():
            print(f'  {sub.name}/model.xml ({xml.stat().st_size/1024:.1f}KB)')
            print(f'  {sub.name}/model.bin ({bin_f.stat().st_size/1e6:.1f}MB)')

print('\nUploaded to:')
for slug, subfolder in [
    ('birdclef2026-exp015-weights', 'r2_ov'),
    ('birdclef2026-exp017-weights', 'r2_ov'),
    ('birdclef2026-exp016-weights', 'r2_ov'),
    ('birdclef2026-exp081-weights', 'r2_ov'),
    ('birdclef2026-exp082-weights', 'r2_ov'),
    ('birdclef2026-exp029-l1-single', 'r3_ov'),
]:
    print(f'  {subfolder}/ → https://www.kaggle.com/datasets/maekeso/{slug}')
